In [35]:
import pandas as pd
import numpy as np
import re
import xarray as xr
import warnings

warnings.filterwarnings('ignore')

In [36]:
df = pd.read_excel('Dane/PositionReport.xlsx', header=[8,9])

In [37]:
czyste_kolumny = []

for col in df.columns:
    czesc_gorna = str(col[0]) if 'Unnamed' not in str(col[0]) else ''
    czesc_dolna = str(col[1]) if 'Unnamed' not in str(col[1]) else ''
    pelna_nazwa = f"{czesc_gorna} {czesc_dolna}".replace('\n', ' ').strip()
    pelna_nazwa = pelna_nazwa.replace('  ', ' ')
    czyste_kolumny.append(pelna_nazwa)
df.columns = czyste_kolumny



In [38]:

def dms_to_dd(dms_str):
    if pd.isna(dms_str):
        return np.nan
    match = re.match(r"(\d+)°\s+([\d.]+)'\s+([NESW])", str(dms_str))
    if not match:
        return np.nan 
    degrees = float(match.group(1))
    minutes = float(match.group(2))
    direction = match.group(3)
    dd = degrees + (minutes / 60.0)
    if direction in ['S', 'W']:
        dd *= -1
        
    return round(dd, 5)
df.columns = df.columns.str.strip()
df['Time'] = pd.to_datetime(df['Time'], format='%d.%m.%Y %H:%M')
df['Lat_dd'] = df['Lat'].apply(dms_to_dd)
df['Lon_dd'] = df['Lon'].apply(dms_to_dd)

df_clean = df.dropna(subset=['Speed [kn]']).copy()
df_clean = df_clean[df_clean['Speed [kn]'] > 0]
print(f"Liczba wierszy przed czyszczeniem: {len(df)}")
print(f"Liczba wierszy po odcięciu postojów: {len(df_clean)}")
display(df_clean[['Time', 'Lat_dd', 'Lon_dd', 'Speed [kn]', 'Course [°]']].head())

Liczba wierszy przed czyszczeniem: 156942
Liczba wierszy po odcięciu postojów: 132042


,Time,Lat_dd,Lon_dd,Speed [kn],Course [°]
0,2023-02-02 00:06:00,56.82667,-0.12833,15.0,357.0
1,2023-02-02 00:21:00,56.88667,-0.12500,15.0,47.0
2,2023-02-02 00:33:00,56.92333,-0.05833,16.0,46.0
3,2023-02-02 00:54:00,56.98667,0.05500,15.0,44.0
4,2023-02-02 01:15:00,57.05000,0.17500,16.0,46.0


In [39]:
print("Bounding Box:")
print(f"Północ (North): {df_clean['Lat_dd'].max()}")
print(f"Południe (South): {df_clean['Lat_dd'].min()}")
print(f"Wschód (East): {df_clean['Lon_dd'].max()}")
print(f"Zachód (West): {df_clean['Lon_dd'].min()}")
print(f"Początek (Od): {df_clean['Time'].min()}")
print(f"Koniec (Do): {df_clean['Time'].max()}")

Bounding Box:
Północ (North): 61.23332
Południe (South): 51.03923
Wschód (East): 11.20088
Zachód (West): -2.12149
Początek (Od): 2023-02-02 00:06:00
Koniec (Do): 2026-02-02 08:34:57


In [40]:
df_clean.head()

,Time,Lat,Lon,Speed [kn],Calculated Speed [kn],Course [°],Distance since last point [nm],Total distance run [nm],Lat_dd,Lon_dd
0,2023-02-02 00:06:00,56° 49.6000' N,000° 07.7000' W,15.0,NaN,357.0,0.000000,0.000000,56.82667,-0.12833
1,2023-02-02 00:21:00,56° 53.2000' N,000° 07.5000' W,15.0,14.4,47.0,3.608126,3.608126,56.88667,-0.12500
2,2023-02-02 00:33:00,56° 55.4000' N,000° 03.5000' W,16.0,15.5,46.0,3.105603,6.713728,56.92333,-0.05833
3,2023-02-02 00:54:00,56° 59.2000' N,000° 03.3000' E,15.0,15.2,44.0,5.318924,12.032653,56.98667,0.05500
4,2023-02-02 01:15:00,57° 03.0000' N,000° 10.5000' E,16.0,15.6,46.0,5.468927,17.501580,57.05000,0.17500


In [41]:

# 1. PRZYGOTOWANIE CZASU I DANYCH ERA5
# Upewniamy się, że czas jest w formacie datetime bez strefy czasowej (kompatybilność z xarray)
df_clean['Time'] = pd.to_datetime(df_clean['Time'], utc=True).dt.tz_localize(None)

sciezka_do_plikow = 'Dane/*.nc'
ds_era5 = xr.open_mfdataset(sciezka_do_plikow, combine='by_coords', engine='netcdf4')

# Konwersja długości geograficznej na format -180 do 180
if ds_era5.longitude.max() > 180:
    ds_era5.coords['longitude'] = (ds_era5.coords['longitude'] + 180) % 360 - 180
    ds_era5 = ds_era5.sortby(ds_era5.longitude)

# Definicja punktów trajektorii
t = xr.DataArray(df_clean['Time'], dims="z")
lat = xr.DataArray(df_clean['Lat_dd'], dims="z")
lon = xr.DataArray(df_clean['Lon_dd'], dims="z")

# 2. INTERPOLACJA WIATRU I FAL
# Wiatr (liniowo)
ds_wind = ds_era5[['u10', 'v10']].interp(valid_time=t, latitude=lat, longitude=lon, method='linear')
ds_wind = ds_wind.compute()

# Fale (nearest + "rozmazanie" brzegów, aby uniknąć NaN przy lądzie)
ds_waves_grid = ds_era5[['swh', 'mwd']]
ds_waves_grid = ds_waves_grid.ffill(dim='longitude', limit=2).bfill(dim='longitude', limit=2)
ds_waves_grid = ds_waves_grid.ffill(dim='latitude', limit=2).bfill(dim='latitude', limit=2)

ds_waves = ds_waves_grid.interp(valid_time=t, latitude=lat, longitude=lon, method='nearest')
ds_waves = ds_waves.compute()

# 3. PRZYPISANIE DO DATAFRAME I OBLICZENIA PODSTAWOWE
df_clean['Wind_U_10m'] = ds_wind['u10'].values
df_clean['Wind_V_10m'] = ds_wind['v10'].values
df_clean['Wind_Speed_m_s'] = np.sqrt(df_clean['Wind_U_10m']**2 + df_clean['Wind_V_10m']**2)

df_clean['Wave_Height_Sig'] = ds_waves['swh'].values
df_clean['Wave_Dir_Mean'] = ds_waves['mwd'].values

# 4. PRO FILLING & INTERPOLATION (Brak NaN-ów pod model)

# Wysokość fali: interpolacja liniowa + fill 0 (brak fali tam, gdzie brak danych)
df_clean['Wave_Height_Sig'] = df_clean['Wave_Height_Sig'].interpolate(method='linear', limit=12).fillna(0)

# Kierunek fali: Interpolacja trygonometryczna (rozbicie na wektory)
dir_rad = np.deg2rad(df_clean['Wave_Dir_Mean'])
df_clean['dir_sin'] = np.sin(dir_rad)
df_clean['dir_cos'] = np.cos(dir_rad)

# Interpolacja składowych (to jest bezpieczniejsze niż interpolacja stopni)
df_clean['dir_sin'] = df_clean['dir_sin'].interpolate(method='linear', limit=12)
df_clean['dir_cos'] = df_clean['dir_cos'].interpolate(method='linear', limit=12)

# Załatanie pozostałych luk w kierunku (na początku/końcu trasy) metodą ffill/bfill
df_clean['dir_sin'] = df_clean['dir_sin'].ffill().bfill()
df_clean['dir_cos'] = df_clean['dir_cos'].ffill().bfill()

# Rekonstrukcja kierunku w stopniach (dla czytelności)
interpolated_dir_rad = np.arctan2(df_clean['dir_sin'], df_clean['dir_cos'])
df_clean['Wave_Dir_Mean'] = (np.rad2deg(interpolated_dir_rad) + 360) % 360

# 5. OPCJA PRO: ML ENCODING (Sin/Cos jako finalne cechy dla modelu)
# Te kolumny są najlepsze dla modeli typu sieci neuronowe czy XGBoost
df_clean['Wave_Dir_Sin'] = df_clean['dir_sin']
df_clean['Wave_Dir_Cos'] = df_clean['dir_cos']

# Usuwamy kolumny pomocnicze
df_clean = df_clean.drop(columns=['dir_sin', 'dir_cos'])

# Wypełnianie braków wietrznych
kolumny_wiatr = ['Wind_U_10m', 'Wind_V_10m', 'Wind_Speed_m_s']
df_clean[kolumny_wiatr] = df_clean[kolumny_wiatr].ffill().bfill()

# 6. PODSUMOWANIE
print("\n--- Połączone i Oczyszczone Dane (Gotowe pod Model) ---")
display(df_clean[['Time', 'Wave_Height_Sig', 'Wave_Dir_Mean', 'Wave_Dir_Sin', 'Wave_Dir_Cos','Wind_U_10m','Wind_V_10m']].head())

print("\nPodsumowanie zbioru (Liczba braków NaN):")
stats = df_clean[['Wave_Height_Sig', 'Wave_Dir_Mean', 'Wind_Speed_m_s', 'Wave_Dir_Sin', 'Wave_Dir_Cos']].isna().sum()
print(stats)

if stats.sum() == 0:
    print("\n✅ Sukces: Wszystkie braki danych zostały usunięte. Dane gotowe do predykcji.")


--- Połączone i Oczyszczone Dane (Gotowe pod Model) ---


,Time,Wave_Height_Sig,Wave_Dir_Mean,Wave_Dir_Sin,Wave_Dir_Cos,Wind_U_10m,Wind_V_10m
0,2023-02-02 00:06:00,2.089863,334.958862,-0.423269,0.906004,7.112340,0.726958
1,2023-02-02 00:21:00,2.152363,334.076050,-0.437178,0.899375,7.243353,0.623888
2,2023-02-02 00:33:00,2.307535,330.844055,-0.487188,0.873297,7.416633,0.372183
3,2023-02-02 00:54:00,2.307535,330.844055,-0.487188,0.873297,7.656569,-0.077478
4,2023-02-02 01:15:00,2.307535,330.844055,-0.487188,0.873297,7.587544,-0.674967



Podsumowanie zbioru (Liczba braków NaN):
Wave_Height_Sig    0
Wave_Dir_Mean      0
Wind_Speed_m_s     0
Wave_Dir_Sin       0
Wave_Dir_Cos       0
dtype: int64

✅ Sukces: Wszystkie braki danych zostały usunięte. Dane gotowe do predykcji.


In [42]:
def dms_to_dd(dms_str):
    """Konwersja współrzędnych z formatu Stopnie Minuty (DMS) na wartości dziesiętne (DD)"""
    if pd.isna(dms_str):
        return np.nan
    match = re.match(r"(\d+)°\s+([\d.]+)'\s+([NESW])", str(dms_str))
    if not match:
        return np.nan 
    degrees = float(match.group(1))
    minutes = float(match.group(2))
    direction = match.group(3)
    dd = degrees + (minutes / 60.0)
    if direction in ['S', 'W']:
        dd *= -1
    return round(dd, 5)

def calculate_awa_improved(df):
    """Obliczanie wiatru pozornego na podstawie danych ze statku i ERA5"""
    df_calc = df.copy()
    
    V_s = df_calc['Speed [kn]'] * 0.514444
    
    course_rad = np.radians(df_calc['Course [°]'])
    df_calc['u_s'] = V_s * np.sin(course_rad)
    df_calc['v_s'] = V_s * np.cos(course_rad)
    
    df_calc['u_a'] = df_calc['Wind_U_10m'] - df_calc['u_s']
    df_calc['v_a'] = df_calc['Wind_V_10m'] - df_calc['v_s']    
    
    df_calc['AWA_north'] = np.degrees(np.arctan2(df_calc['u_a'], df_calc['v_a']))   
    df_calc['AWA_relative'] = (df_calc['AWA_north'] - df_calc['Course [°]'] + 180) % 360 - 180   
    
    df_calc['Apparent_Wind_Speed_m_s'] = np.sqrt(df_calc['u_a']**2 + df_calc['v_a']**2)
    
    return df_calc
    

In [44]:
import glob
import os

# ==========================================
# INTEGRACJA DANYCH PRĄDÓW I POZIOMU MORZA
# ==========================================
print("Wczytywanie danych prądów morskich (currents) i poziomu morza (sea_level)...")

# Wczytanie danych prądów (u, v komponenty)
currents_path = 'Dane/currents/*.nc'
ds_currents = xr.open_mfdataset(currents_path, combine='by_coords', engine='netcdf4')

# Wczytanie danych poziomu morza
# SSH dostępne tylko dla 2025+, ale pływ się powtarza
sea_level_path = 'Dane/sea_level/*.nc'
ds_ssh_raw = xr.open_mfdataset(sea_level_path, combine='by_coords', engine='netcdf4')

# Konwersja długości geograficznej na format -180 do 180 (jeśli potrzeba)
if ds_currents.longitude.max() > 180:
    ds_currents.coords['longitude'] = (ds_currents.coords['longitude'] + 180) % 360 - 180
    ds_currents = ds_currents.sortby(ds_currents.longitude)

if ds_ssh_raw.longitude.max() > 180:
    ds_ssh_raw.coords['longitude'] = (ds_ssh_raw.coords['longitude'] + 180) % 360 - 180
    ds_ssh_raw = ds_ssh_raw.sortby(ds_ssh_raw.longitude)

# Rozszerzenie SSH na wszystkie lata (pływ się powtarza co rok)
# Wyodrębniamy dzień roku z danych SSH i mapujemy go na wszystkie lata w danych prądów
ds_currents_range = pd.to_datetime(ds_currents.time.values)
ssh_raw_range = pd.to_datetime(ds_ssh_raw.time.values)

# Funkcja do mapowania daty na dzień roku i znalezienia odpowiadającej daty w SSH
def map_ssh_to_all_years(date_range, ssh_date_range):
    """
    Mapuje daty na dzień roku i znajduje najbliższą datę SSH dla każdego dnia
    """
    mapped_times = []
    
    for target_date in date_range:
        target_day_of_year = target_date.dayofyear
        target_is_leap = pd.Timestamp(target_date.year, 12, 31).dayofyear == 366
        
        # Znajdź datę SSH z takim samym dniem roku
        best_match = None
        min_diff = float('inf')
        
        for ssh_date in ssh_date_range:
            ssh_day_of_year = ssh_date.dayofyear
            diff = abs(target_day_of_year - ssh_day_of_year)
            
            if diff < min_diff:
                min_diff = diff
                best_match = ssh_date
        
        mapped_times.append(best_match)
    
    return np.array(mapped_times, dtype='datetime64[ns]')

# Mapowanie SSH na wszystkie czasy w danych prądów
ssh_mapped_times = map_ssh_to_all_years(ds_currents_range, ssh_raw_range)

# Wybierz topwarstwę głębokości dla prądów (depth=0)
ds_currents_surface = ds_currents.isel(depth=0)

print(f"Dane prądów (powierzchnia): {len(ds_currents_range)} czasów")
print(f"Dane SSH (oryginalne): {len(ssh_raw_range)} czasów")
print("Mapowanie SSH na wszystkie lata zakończone.")

Wczytywanie danych prądów morskich (currents) i poziomu morza (sea_level)...
Dane prądów (powierzchnia): 875 czasów
Dane SSH (oryginalne): 365 czasów
Mapowanie SSH na wszystkie lata zakończone.


In [45]:
# ==========================================
# INTERPOLACJA PRĄDÓW I SSH NA TRAJEKTORIĘ STATKU
# ==========================================
print("\nInterpolacja prądów morskich...")

t_currents = xr.DataArray(df_clean['Time'], dims="z")
lat_currents = xr.DataArray(df_clean['Lat_dd'], dims="z")
lon_currents = xr.DataArray(df_clean['Lon_dd'], dims="z")

# Interpolacja prądów (liniowa) - powierzchnia (depth=0)
ds_currents_interp = ds_currents_surface[['uo', 'vo']].interp(
    time=t_currents, 
    latitude=lat_currents, 
    longitude=lon_currents, 
    method='linear'
)
ds_currents_interp = ds_currents_interp.compute()

df_clean['Current_U_m_s'] = ds_currents_interp['uo'].values
df_clean['Current_V_m_s'] = ds_currents_interp['vo'].values
df_clean['Current_Speed_m_s'] = np.sqrt(df_clean['Current_U_m_s']**2 + df_clean['Current_V_m_s']**2)

# Interpolacja SSH na trajektorię statku (używając zmapowanych czasów)
print("Interpolacja poziomu morza (SSH)...")

# Budowa dataset SSH z rozszerzonymi czasami
ssh_extended_list = []
for orig_time, mapped_time in zip(pd.to_datetime(ds_currents.time.values), ssh_mapped_times):
    # Wczytaj wartość SSH dla zmapowanego czasu
    ssh_val = ds_ssh_raw.sel(time=mapped_time, method='nearest')
    ssh_extended_list.append(ssh_val)

# Połącz wszystkie dane SSH
ds_ssh = xr.concat(ssh_extended_list, dim='time')
ds_ssh['time'] = ('time', pd.to_datetime(ds_currents.time.values))

# Interpolacja SSH na punkty trajektorii
ds_ssh_interp = ds_ssh[['zos']].interp(
    time=t_currents,
    latitude=lat_currents,
    longitude=lon_currents,
    method='nearest'
)
ds_ssh_interp = ds_ssh_interp.compute()

df_clean['Sea_Level_SSH_m'] = ds_ssh_interp['zos'].values

# Wypełnienie braków danych (ffill/bfill)
kolumny_do_fillna = ['Current_U_m_s', 'Current_V_m_s', 'Current_Speed_m_s', 'Sea_Level_SSH_m']
for col in kolumny_do_fillna:
    df_clean[col] = df_clean[col].ffill().bfill()

print("\n--- Dane Prądów i SSH (Pierwsze 5 wierszy) ---")
display(df_clean[['Time', 'Current_U_m_s', 'Current_V_m_s', 'Current_Speed_m_s', 'Sea_Level_SSH_m']].head())

# Sprawdzenie braków
print("\nPodsumowanie zbioru (Brakujące wartości):")
stats = df_clean[['Current_U_m_s', 'Current_V_m_s', 'Current_Speed_m_s', 'Sea_Level_SSH_m']].isna().sum()
print(stats)

if stats.sum() == 0:
    print("\n✅ Sukces: Wszystkie dane prądów i SSH zostały zintegrowane.")


Interpolacja prądów morskich...
Interpolacja poziomu morza (SSH)...

--- Dane Prądów i SSH (Pierwsze 5 wierszy) ---


,Time,Current_U_m_s,Current_V_m_s,Current_Speed_m_s,Sea_Level_SSH_m
0,2023-02-02 00:06:00,0.00787,0.030347,0.031351,-0.288
1,2023-02-02 00:21:00,0.00787,0.030347,0.031351,-0.288
2,2023-02-02 00:33:00,0.00787,0.030347,0.031351,-0.288
3,2023-02-02 00:54:00,0.00787,0.030347,0.031351,-0.288
4,2023-02-02 01:15:00,0.00787,0.030347,0.031351,-0.288



Podsumowanie zbioru (Brakujące wartości):
Current_U_m_s        0
Current_V_m_s        0
Current_Speed_m_s    0
Sea_Level_SSH_m      0
dtype: int64

✅ Sukces: Wszystkie dane prądów i SSH zostały zintegrowane.


In [46]:
# ==========================================
# 4. OBLICZANIE WIATRU POZORNEGO DLA ROTORA
# ==========================================
print("Obliczanie parametrów wiatru pozornego...")
df_clean = calculate_awa_improved(df_clean)


Obliczanie parametrów wiatru pozornego...


In [47]:
# ==========================================
# 5. PODSUMOWANIE I ZAPIS
# ==========================================
print("\n--- Sprawdzenie końcowe (Brakujące wartości) ---")
stats = df_clean[['Wave_Height_Sig', 'Wave_Dir_Mean', 'Wind_Speed_m_s', 'Current_Speed_m_s', 'Sea_Level_SSH_m', 'AWA_relative']].isna().sum()
print(stats.to_string())

if stats.sum() == 0:
    print("\n✅ Sukces: Brak pustych wartości. Dane są idealnie przygotowane do modelowania.")
else:
    print("\n⚠️ Uwaga: Zostały puste wartości, zostały uzupełnione metodą ffill/bfill.")



--- Sprawdzenie końcowe (Brakujące wartości) ---
Wave_Height_Sig      0
Wave_Dir_Mean        0
Wind_Speed_m_s       0
Current_Speed_m_s    0
Sea_Level_SSH_m      0
AWA_relative         0

✅ Sukces: Brak pustych wartości. Dane są idealnie przygotowane do modelowania.


In [51]:
import io
from scipy.interpolate import LinearNDInterpolator
print("Interpolacja mocy rotorów z wykresu biegunowego...")

rotor_data_csv = """Angle,Wind_Speed,Power_kW
0,1,-2
45,1,4
90,1,14
135,1,10
180,1,-3
0,2,-8
45,2,16
90,2,56
135,2,40
180,2,-12
0,3,-18
45,3,36
90,3,126
135,3,90
180,3,-27
0,4,-32
45,4,64
90,4,224
135,4,160
180,4,-48
0,5,-50
45,5,100
90,5,350
135,5,250
180,5,-75
0,6,-68
45,6,140
90,6,450
135,6,340
180,6,-100
0,7,-85
45,7,185
90,7,560
135,7,440
180,7,-130
0,8,-105
45,8,235
90,8,660
135,8,530
180,8,-160
0,9,-125
45,9,290
90,9,750
135,9,610
180,9,-190
0,10,-150
45,10,350
90,10,850
135,10,700
180,10,-220
0,11,-175
45,11,410
90,11,960
135,11,780
180,11,-250
0,12,-200
45,12,470
90,12,1080
135,12,870
180,12,-285
0,13,-225
45,13,530
90,13,1200
135,13,950
180,13,-320
0,14,-250
45,14,590
90,14,1330
135,14,1030
180,14,-350
0,15,-280
45,15,650
90,15,1450
135,15,1100
180,15,-380
0,16,-310
45,16,710
90,16,1550
135,16,1180
180,16,-410
0,17,-340
45,17,770
90,17,1650
135,17,1260
180,17,-440
0,18,-370
45,18,830
90,18,1750
135,18,1340
180,18,-470
0,19,-400
45,19,890
90,19,1850
135,19,1420
180,19,-500
0,20,-430
45,20,950
90,20,1950
135,20,1500
180,20,-530
0,21,-460
45,21,1010
90,21,2050
135,21,1570
180,21,-560
0,22,-490
45,22,1070
90,22,2150
135,22,1640
180,22,-590
0,23,-520
45,23,1130
90,23,2220
135,23,1710
180,23,-620
0,24,-550
45,24,1190
90,24,2280
135,24,1780
180,24,-650
0,25,-580
45,25,1250
90,25,2350
135,25,1850
180,25,-680"""

df_rotor_map = pd.read_csv(io.StringIO(rotor_data_csv))
points = df_rotor_map[['Angle', 'Wind_Speed']].values
values = df_rotor_map['Power_kW'].values
rotor_interpolator = LinearNDInterpolator(points, values)

def calculate_total_rotor_power(row):
    wind_speed = row['Apparent_Wind_Speed_m_s']
    
    # WARUNEK - WIATR POZORNY POWYŻEJ 40 M/S - ROTORY NIE PRACUJĄ
    if wind_speed > 40:
        return 0.0
    
    # Pobieramy wartość absolutną kąta wiatru pozornego (bo lewa i prawa burta dają ten sam ciąg)
    wind_angle = abs(row['AWA_relative'])
    
    # Przeliczenie dla symetrii 0-180 (jeśli kąt był > 180 z innej przyczyny matematycznej)
    wind_angle = wind_angle % 360
    if wind_angle > 180:
        wind_angle = 360 - wind_angle
        
    # Zamrożenie wartości granicznych (clipping), żeby interpolator nie zwracał NaN
    wind_speed = np.clip(wind_speed, 1, 25)
    
    # Pobranie mocy dla 1 rotora
    power_1_rotor = rotor_interpolator(wind_angle, wind_speed)
    
    if np.isnan(power_1_rotor):
        power_1_rotor = 0.0
        
    # Statek posiada dwa rotory (mnożymy * 2)
    total_power = power_1_rotor * 2
    
    # Ochrona przed oporem aerodynamicznym (ujemną mocą)
    if total_power < 0:
        total_power = 0
        
    return total_power

df_clean['Total_Rotor_Power_kW'] = df_clean.apply(calculate_total_rotor_power, axis=1)


# ==========================================
# 5. PODSUMOWANIE I ZAPIS
# ==========================================
print("\n--- Sprawdzenie końcowe (Brakujące wartości) ---")
stats = df_clean[['Wave_Height_Sig', 'Wave_Dir_Mean', 'Wind_Speed_m_s', 'Current_Speed_m_s', 'Sea_Level_SSH_m', 'AWA_relative']].isna().sum()
print(stats.to_string())

print(f"\n--- Warunki pracy rotorów ---")
print(f"Przypadki z wiatrem pozornym > 40 m/s (ROTORY WYŁĄCZONE): {(df_clean['Apparent_Wind_Speed_m_s'] > 40).sum()}")
print(f"Max prędkość wiatru pozornego: {df_clean['Apparent_Wind_Speed_m_s'].max():.2f} m/s")
print(f"Min prędkość wiatru pozornego: {df_clean['Apparent_Wind_Speed_m_s'].min():.2f} m/s")

if stats.sum() == 0:
    print("\n✅ Sukces: Brak pustych wartości. Dane są idealnie przygotowane do modelowania.")

Interpolacja mocy rotorów z wykresu biegunowego...

--- Sprawdzenie końcowe (Brakujące wartości) ---
Wave_Height_Sig      0
Wave_Dir_Mean        0
Wind_Speed_m_s       0
Current_Speed_m_s    0
Sea_Level_SSH_m      0
AWA_relative         0

--- Warunki pracy rotorów ---
Przypadki z wiatrem pozornym > 40 m/s (ROTORY WYŁĄCZONE): 0
Max prędkość wiatru pozornego: 22.74 m/s
Min prędkość wiatru pozornego: 0.02 m/s

✅ Sukces: Brak pustych wartości. Dane są idealnie przygotowane do modelowania.


In [52]:
output_path = 'dane_gotowe_rotor.csv'
kolumny_do_zapisu = [
    'Time', 'Lat_dd', 'Lon_dd', 'Speed [kn]', 'Course [°]', 
    'Wind_Speed_m_s', 'Wind_U_10m', 'Wind_V_10m', 
    'Wave_Height_Sig', 'Wave_Dir_Mean', 'Wave_Dir_Sin', 'Wave_Dir_Cos',
    'Current_U_m_s', 'Current_V_m_s', 'Current_Speed_m_s',
    'Sea_Level_SSH_m',
    'AWA_relative', 'Apparent_Wind_Speed_m_s', 'Total_Rotor_Power_kW'
]

print(f"\nSummary of data with wind speed limits:")
print(f"Apparent wind speed > 40 m/s: {(df_clean['Apparent_Wind_Speed_m_s'] > 40).sum()} cases (rotors OFF)")

print(f"\nZapisano gotowy zbiór ({len(df_clean)} wierszy) do pliku: {output_path}")
df_clean[kolumny_do_zapisu].to_csv(output_path, index=False)


Summary of data with wind speed limits:
Apparent wind speed > 40 m/s: 0 cases (rotors OFF)

Zapisano gotowy zbiór (132042 wierszy) do pliku: dane_gotowe_rotor.csv


In [53]:
gotowe = pd.read_csv('dane_gotowe_rotor.csv')
gotowe.head()

,Time,Lat_dd,Lon_dd,Speed [kn],Course [°],Wind_Speed_m_s,Wind_U_10m,Wind_V_10m,Wave_Height_Sig,Wave_Dir_Mean,Wave_Dir_Sin,Wave_Dir_Cos,Current_U_m_s,Current_V_m_s,Current_Speed_m_s,Sea_Level_SSH_m,AWA_relative,Apparent_Wind_Speed_m_s,Total_Rotor_Power_kW
0,2023-02-02 00:06:00,56.82667,-0.12833,15.0,357.0,7.149395,7.112340,0.726958,2.089863,334.95886,-0.423269,0.906004,0.00787,0.030347,0.031351,-0.288,135.878082,10.256776,1400.887463
1,2023-02-02 00:21:00,56.88667,-0.12500,15.0,47.0,7.270172,7.243353,0.623888,2.152363,334.07605,-0.437178,0.899375,0.00787,0.030347,0.031351,-0.288,113.972876,4.906957,570.006967
2,2023-02-02 00:33:00,56.92333,-0.05833,16.0,46.0,7.425966,7.416633,0.372183,2.307535,330.84406,-0.487188,0.873297,0.00787,0.030347,0.031351,-0.288,118.368708,5.550920,673.082473
3,2023-02-02 00:54:00,56.98667,0.05500,15.0,44.0,7.656961,7.656569,-0.077478,2.307535,330.84406,-0.487188,0.873297,0.00787,0.030347,0.031351,-0.288,113.806743,6.078720,800.929924
4,2023-02-02 01:15:00,57.05000,0.17500,16.0,46.0,7.617507,7.587544,-0.674967,2.307535,330.84406,-0.487188,0.873297,0.00787,0.030347,0.031351,-0.288,119.388324,6.606439,884.550049
